<a href="https://colab.research.google.com/github/KULL-Centre/ColabCALVADOS/blob/main/CALVADOS_simulate_and_reweight.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CALVADOS Colab: Simulations of IDRs and MDPs
This Colab notebook enables running molecular dynamics (MD) simulations of intrinsically disordered proteins (IDPs) and multi-domain proteins (MDPs) and to study their conformational ensembles [1,2,3].

MD simulations employ the coarse-grained force fields CALVADOS 2 or 3 [1,2], where C$_\alpha$ and center-of-mass (COM) representation will be used for IDPs and MDPs, respectively.

Simulations are run using OpenMM [4] based on user-defined `temperature`, `ionic strength`, `pH`, `sequence`, and `domain boundary` (only for MDPs). Detailed instructions can be found in Cell 2.

Coarse-grained trajectories are converted to all-atom trajectories using cg2all [5].

### Usage
We recommend running MD simulations run on a single GPU. To enable GPU select `Runtime` from the menu, then `Change runtime type` and select `GPU`.

Note: Cells for preliminary operations should be executed one by one to prevent crashes. This notebook uses condacolab, whose installation will cause a kernel restart. Because of this, a crash warning will happen during preliminary operations if you execute all cells at once.

### References
If you use this notebook, you may consider citing [1] for simulations using CALVADOS 2, [2] for simulations using CALVADOS 3, [3] for the machine learning predictor of scaling exponents, [4] for OpenMM, and [5] for cg2all.

1. G. Tesei and K. Lindorff-Larsen __Improved predictions of phase behaviour of intrinsically disordered proteins by tuning the interaction range [version 2; peer review: 2 approved]__ _Open Res Eur._ 2023 2(94) DOI: https://doi.org/10.12688/openreseurope.14967.2

2. F. Cao, S. von Bülow, G. Tesei, and K. Lindorff‐Larsen (2024). __A coarse‐grained model for disordered and multi‐domain proteins.__ _Protein Sci._ 33(11):e5172 DOI: https://doi.org/10.1002/pro.5172

3. G. Tesei, A. I. Trolle, N. Jonsson, J. Betz, F. E. Knudsen, F. Pesce, K. E. Johansson, K. Lindorff-Larsen __Conformational ensembles of the human intrinsically disordered proteome__ _Nature._ 2024 626:897–904 2023.05.08.539815 DOI: https://doi.org/10.1038/s41586-023-07004-5

4. P. Eastman, J. Swails, J. D. Chodera et al. __OpenMM 7: Rapid development of high performance algorithms for molecular dynamics__ _PLoS Comput Biol._ 2017 13(7):e1005659 DOI: https://doi.org/10.1371/journal.pcbi.1005659

5. L. Heo and M. Feig __One particle per residue can describe all-atom protein structures.__ _Structure_, 2024 32(1):97–111.e6

In [1]:
# @title 1. Set the environment for simulation

# @markdown Attention! A prompt of "Your session crashed for an unknown reason" will appear after running this block. This is required for all packages to work properly. Please ignore it and move on.

import subprocess
import locale
locale.getpreferredencoding = lambda: "UTF-8"
subprocess.run('pip install py3Dmol'.split())
subprocess.run('pip install MDAnalysis'.split())
subprocess.run('pip install mdtraj'.split())
subprocess.run('pip install -q condacolab'.split())
import condacolab
condacolab.install()
subprocess.run('mamba install openmm -c conda-forge --yes'.split())
subprocess.run('pip install wget'.split())
subprocess.run('pip install fastprogress'.split())
subprocess.run('pip install kneed==0.5.0'.split())
subprocess.run('mamba install pandas'.split())
subprocess.run('mamba install pydantic'.split())

⏬ Downloading https://github.com/jaimergp/miniforge/releases/download/24.11.2-1_colab/Miniforge3-colab-24.11.2-1_colab-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:08
🔁 Restarting kernel...


CompletedProcess(args=['mamba', 'install', 'pydantic'], returncode=1)

In [ ]:
# @title 2. Configure your protein

#@markdown In [data/SAXS/exp_conditions.csv](https://github.com/KULL-Centre/ColabCALVADOS/blob/main/data/SAXS/exp_conditions.csv) you can find __temperature__, __ionic strength__, __pH__, __domain boundaries__, and __sequence__ information for SAXS measurements of 13 IDPs and 13 MDPs (see DOI: 10.1016/j.bpj.2022.12.013, 10.1002/pro.5172).

#@markdown In [data/SAXS](https://github.com/KULL-Centre/ColabCALVADOS/tree/main/data/SAXS), you can find PDB files (`.pdb`) for 13 MDPs.

#@markdown Choose the protein you want to work with and download its `.pdb` file (for MDPs). You can also use your own PDB for the protein of your choice.

import numpy as np
import os
import shutil
import ipywidgets as widgets
import warnings
import wget
import yaml
from google.colab import files

warnings.filterwarnings('ignore')

# @markdown WARNING: Tick this box ONLY if you want to run the Ubq4 example.
example = False  #@param {type:"boolean"}
# @markdown If `example` is checked, all input below will be ignored and the simulation will use fixed Ubq4 (MDP) settings.

isIDP = False #@param {type:"boolean"}
name = "Ubq4" #@param {type:"string"}
sequence = "MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGGMQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGGMQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGGMQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG" #@param {type:"string"}
L = int(np.ceil((len(sequence) - 1) * 0.38 + 4))
if L > 900:  # nm
    raise Exception("Too large box! Floats might overflow in PDB file. You can try to modify the 'L' parameter in this block yourself.")
temperature = 293 #@param {type:"number"}
ionic_strength = 0.33 #@param {type:"number"}
pH = 8.0 #@param {type:"number"}
cutoff = 2.0  # nm
#@markdown <i>*Units: temperature [K], ionic strength [M]<i>

#@markdown <b>Define domain boundary -- ignore this if `isIDP` is toggled on:

#@markdown Residue ranges are delimited by `-`, e.g. residues 1 to 200 are `1-200`;

#@markdown Separate domains are delimited by `,`, e.g. `1-200,201-400`;

#@markdown Discontinuous segments are delimited by `_`, e.g. `1-200,201-300_350-400`.
domain_boundary = "1-72,77-148,153-224,229-300" #@param {type:"string"}
input_pae = False
k_restraint = 700  # unit:KJ/(mol*nm^2)
eps_factor = 0.2
CALVADOS_version = 3
discard_first_nframes = 10

if example:
    name = "Ubq4"
    isIDP = False
    sequence = "MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGGMQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGGMQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGGMQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG"
    L = int(np.ceil((len(sequence) - 1) * 0.38 + 4))
    temperature = 293
    ionic_strength = 0.33
    pH = 8.0
    domain_boundary = "1-72,77-148,153-224,229-300"

use_pdb = False if isIDP else True
use_hnetwork = False if isIDP else True
use_ssdomains = False if isIDP else True

if not isIDP:
    domain_resNum = []  # start from 1
    for domain in domain_boundary.split(","):
        domain = domain.split("_")
        if len(domain) == 1:  # no break point
            domain_resNum.append(list(range(int(domain[0].split('-')[0]), int(domain[0].split('-')[1]) + 1)))
        else:
            tmp_list = []
            for subdomain in domain:
                tmp_list += list(range(int(subdomain.split('-')[0]), int(subdomain.split('-')[1]) + 1))
            domain_resNum.append(tmp_list)
else:
    domain_boundary = None
    domain_resNum = None
    path2pdb = ''

print(f"{'IDP' if isIDP else 'MDP'} name: {name}")
print(f"sequence: {sequence}")
print(f"simulation box: [{L}, {L}, {L}] nm")
print(f"temperature: {temperature} K")
print(f"ionic strength: {ionic_strength} M")
print(f"pH: {pH}")
if not isIDP:
    print(f"There are {len(domain_resNum)} restrained domains (1-based): ")
    for d in domain_resNum:
        print(d)
if not os.path.isdir(f"{name}"):
    os.system(f"mkdir -p {name}")
    os.system(f"mkdir -p {name}_tmp")

In [ ]:
# @title 3. Upload a PDB file for MDPs (skip if your protein is an IDP)

# @markdown The residues in the PDB file need to be listed in the same order as they appear in the protein sequence. This ensures the PDB file is consistent with the definition of the folded domains.

# @markdown If you are using the `example`, the PDB file is [here](https://github.com/KULL-Centre/ColabCALVADOS/blob/main/data/SAXS/THB_C2.pdb). If you are using your own pdb file, the scripts will rename your file as `${protein_name}.pdb` after upload.

if not isIDP:
    uploaded = files.upload()
    if len(list(uploaded.keys())) != 1:
        raise Exception("Please upload just one pdb file")
    for key in uploaded.keys():
        os.system(f"mv {key} {name}/{name}.pdb")
    path2pdb = f"{name}/{name}.pdb"
    print(f"{path2pdb} successfully uploaded.")
else:
    print("No need to upload a PDB file")

In [ ]:
# @title 4. Set simulation time

#@markdown The default option “AUTO” will set the simulation time depending on the total sequence length of the IDR(s). The longer the IDR, the larger the ensemble of conformations the protein can adopt. Moreover, the reconfiguration time of IDRs increases with increasing sequence length. Therefore, longer sequences will require more sampling. Typical simulation times range from ca. 5 min (a 71-ns-long simulation of an IDR of 70 residues), 7 min (71-ns-long simulation of an IDR of 140 residues), to 34 min for a 373-ns-long simulation of an IDR of 351 residues.
Simulation_time = "AUTO" #@param {type:"string"}
#@markdown <i>*Units: Simulation_time [ns]<i>
N_res = len(sequence)
if not isIDP:
    domain_len = 0
    for domain in domain_resNum:
        domain_len += len(domain)
    N_res -= domain_len

N_save = 7000 if N_res < 150 else int(np.ceil(3e-4*N_res**2)*1000)

if Simulation_time == "AUTO":
    N_steps = 1010 * int(N_save)
    print(f'AUTO simulation length selected. Running for {N_steps*0.01/1000} ns, {N_steps} steps in total.')
else:
    N_steps = int(float(Simulation_time)*1000/0.01)
    print(f'Simulation length assigned. Running for {Simulation_time} ns, {N_steps} steps in total.')

nframes=N_steps // N_save
config_sim_data = dict(
temp=float(temperature), ionic=float(ionic_strength),
pH=float(pH), cutoff=cutoff,
L=L, wfreq=int(N_save), use_pdb=use_pdb,
path2pdb=path2pdb, use_hnetwork=use_hnetwork, domain_resNum=domain_resNum,
use_ssdomains=use_ssdomains, input_pae=input_pae,
k_restraint=k_restraint, protein_name=name,
gpu_id=0, N_res=int(N_res),
CoarseGrained="COM", isIDP=isIDP,
eps_factor=eps_factor,
CALVADOS_version=CALVADOS_version, seq=list(sequence), steps=N_steps,
discard_first_nframes=discard_first_nframes,
nframes=nframes)
yaml.dump(config_sim_data, open(f'{name}/config_sim.yaml', 'w'))

In [ ]:
#@title <b><font color='#A79AB2'>MD simulation toolbox</font></b>
import time
from openmm import app
from simtk import openmm, unit
import pandas as pd
import MDAnalysis as mda
import mdtraj as md
from fastprogress import master_bar, progress_bar

def geometry_from_pdb(protein_name, pdb, CoarseGrained="CA", ssdomains=None):
    """ positions in nm """
    u = mda.Universe(pdb)
    ini_CA = u.select_atoms("name CA")
    ini_CA.write(f"{protein_name}/ini_CA.pdb")
    if CoarseGrained == "CA":
        cas = u.select_atoms('name CA')  # in-place
        center_of_mass = cas.center_of_mass()
        cas.translate(-center_of_mass)
        pos = cas.positions / 10.  # nm, shape: (n, 3)
    elif CoarseGrained == "COM":  # COM
        pos = []
        resSeqinDomain = []
        for ssdomain in ssdomains:
            resSeqinDomain += ssdomain
        for i in range(len(u.residues)):  # i starts from 0
            if i+1 in resSeqinDomain:  # COM
                pos.append(u.residues[i].atoms.center_of_mass())
            else:  # CA
                pos.append(np.squeeze(u.residues[i].atoms.select_atoms("name CA").positions))
        center_of_mass = u.atoms.center_of_mass()
        pos = (np.array(pos) - center_of_mass) / 10.  # nm

    return pos, center_of_mass/10.  # nm

def genParamsDH(df,protein_name,prot,temp):
    kT = 8.3145*temp*1e-3
    fasta = prot.fasta.copy()
    r = df.copy()
    q = 1.0

    r.loc['H','q'] = q / ( 1 + 10**(prot.pH-6) )
    r.loc['X'] = r.loc[fasta[0]]
    r.loc['Z'] = r.loc[fasta[-1]]
    fasta[0] = 'X'
    fasta[-1] = 'Z'
    r.loc['X','q'] = r.loc[prot.fasta[0],'q'] + q
    r.loc['Z','q'] = r.loc[prot.fasta[-1],'q'] - q
    # Calculate the prefactor for the Yukawa potential
    fepsw = lambda T : 5321/T+233.76-0.9297*T+0.1417*1e-2*T*T-0.8292*1e-6*T**3
    epsw = fepsw(temp)
    lB = 1.6021766**2/(4*np.pi*8.854188*epsw)*6.022*1000/kT
    yukawa_eps = [r.loc[a].q*np.sqrt(lB*kT) for a in fasta]
    # Calculate the inverse of the Debye length
    yukawa_kappa = np.sqrt(8*np.pi*lB*prot.ionic*6.022/10)
    return yukawa_eps, yukawa_kappa

def genParamsLJ(df,protein_name,prot):
    fasta = prot.fasta.copy()
    r = df.copy()
    r.loc['X'] = r.loc[fasta[0]]
    r.loc['Z'] = r.loc[fasta[-1]]
    r.loc['X','MW'] += 2
    r.loc['Z','MW'] += 16
    fasta[0] = 'X'
    fasta[-1] = 'Z'
    types = list(np.unique(fasta))
    MWs = [r.loc[a,'MW'] for a in types]
    lj_eps = prot.eps_factor*4.184
    return lj_eps, fasta, types, MWs

def load_parameters(protein_name, CALVADOS_version, output=False):
    wget.download(f"https://github.com/KULL-Centre/ColabCALVADOS/raw/main/data/residues_CALVADOS{CALVADOS_version:d}.csv")
    os.system(f"mv residues_CALVADOS{CALVADOS_version}.csv {protein_name}/residues.csv")
    residues = pd.read_csv(f'{protein_name}/residues.csv').set_index('one', drop=False)
    if output:
        print("Lambda used:\n", residues.lambdas)
    return residues

def p2c(r, phi):
    """
    polar to cartesian
    """
    return (r * np.cos(phi), r * np.sin(phi))

def xy_spiral_array(n, delta=0, arc=.38, separation=.7):
    """
    create points on an Archimedes' spiral
    with `arc` giving the length of arc between two points
    and `separation` giving the distance between consecutive
    turnings
    """
    r = arc
    b = separation / (2 * np.pi)
    phi = float(r) / b
    coords = []
    for i in range(n):
        coords.append(list(p2c(r, phi))+[0])
        phi += float(arc) / r
        r = b * phi
    return np.array(coords)+delta

def build_topology(fasta,n_chains=1):
    # build CG topology
    top = md.Topology()
    for i_chain in range(n_chains):
        chain = top.add_chain()
        for resname in fasta:
            residue = top.add_residue(resname, chain)
            top.add_atom(resname, element=md.element.carbon, residue=residue)
        for i in range(chain.n_atoms-1):
            top.add_bond(chain.atom(i),chain.atom(i+1))
    return top

def build_box(Lx,Ly,Lz):
    # set box vectors
    a = unit.Quantity(np.zeros([3]), unit.nanometers)
    a[0] = Lx * unit.nanometers
    b = unit.Quantity(np.zeros([3]), unit.nanometers)
    b[1] = Ly * unit.nanometers
    c = unit.Quantity(np.zeros([3]), unit.nanometers)
    c[2] = Lz * unit.nanometers
    return a, b, c

def add_particles(system, residues, prot, n_chains=1):
    for i_chain in range(n_chains):
        system.addParticle((residues.loc[prot.fasta[0]].MW+2)*unit.amu)
        for a in prot.fasta[1:-1]:
            system.addParticle(residues.loc[a].MW*unit.amu)
        system.addParticle((residues.loc[prot.fasta[-1]].MW+16)*unit.amu)
    return system

def euclidean(a_matrix, b_matrix):
    # Using matrix operation to calculate Euclidean distance
    d1 = -2 * np.dot(a_matrix, b_matrix.T)
    d2 = np.sum(np.square(a_matrix), axis=1, keepdims=True)
    d3 = np.sum(np.square(b_matrix), axis=1)
    dist = np.sqrt(d1 + d2 + d3)
    return dist

def set_interactions(system, residues, prot, lj_eps, cutoff, yukawa_kappa, yukawa_eps, N, n_chains=1,
                     CoarseGrained="CA", dismatrix=None, isIDP=True, domain_resNum=None):
    hb = openmm.openmm.HarmonicBondForce()
    # interactions
    energy_expression = 'select(step(r-2^(1/6)*s),4*eps*l*((s/r)^12-(s/r)^6-shift),4*eps*((s/r)^12-(s/r)^6-l*shift)+eps*(1-l))'
    ah = openmm.openmm.CustomNonbondedForce(energy_expression + '; s=0.5*(s1+s2); l=0.5*(l1+l2); shift=(0.5*(s1+s2)/rc)^12-(0.5*(s1+s2)/rc)^6')
    ah.addGlobalParameter('eps', lj_eps * unit.kilojoules_per_mole)
    ah.addGlobalParameter('rc', float(cutoff) * unit.nanometer)
    ah.addPerParticleParameter('s')
    ah.addPerParticleParameter('l')
    print('rc', cutoff * unit.nanometer)

    yu = openmm.openmm.CustomNonbondedForce('q*(exp(-kappa*r)/r-shift); q=q1*q2')
    yu.addGlobalParameter('kappa', yukawa_kappa / unit.nanometer)
    yu.addGlobalParameter('shift', np.exp(-yukawa_kappa * 4.0) / 4.0 / unit.nanometer)
    yu.addPerParticleParameter('q')

    for j in range(n_chains):
        begin = j * N  # 0
        end = j * N + N  # n_residues
        for a, e in zip(prot.fasta, yukawa_eps):
            yu.addParticle([e * unit.nanometer * unit.kilojoules_per_mole])
            ah.addParticle([residues.loc[a].sigmas * unit.nanometer, residues.loc[a].lambdas * unit.dimensionless])

        for i in range(begin, end - 1):  # index starts from 0
            if CoarseGrained=="CA" or isIDP:
                hb.addBond(i, i + 1, 0.38 * unit.nanometer, 8033.0 * unit.kilojoules_per_mole / (unit.nanometer ** 2))
            else:  # COM or COE
                ssdomains = domain_resNum
                resSeqinDomain = []
                for ssdomain in ssdomains:
                    resSeqinDomain += ssdomain
                if i+1 in resSeqinDomain or i+1+1 in resSeqinDomain:
                    hb.addBond(i, i + 1, dismatrix[i][i+1] * unit.nanometer, 8033.0 * unit.kilojoules_per_mole / (unit.nanometer ** 2))
                else:
                    hb.addBond(i, i + 1, 0.38 * unit.nanometer, 8033.0 * unit.kilojoules_per_mole / (unit.nanometer ** 2))
            yu.addExclusion(i, i + 1)
            ah.addExclusion(i, i + 1)

    # All Forces in a single force group must use the same cutoff distance
    yu.setForceGroup(0)
    ah.setForceGroup(1)
    yu.setNonbondedMethod(openmm.openmm.CustomNonbondedForce.CutoffPeriodic)
    ah.setNonbondedMethod(openmm.openmm.CustomNonbondedForce.CutoffPeriodic)
    hb.setUsesPeriodicBoundaryConditions(True)
    yu.setCutoffDistance(4 * unit.nanometer)
    ah.setCutoffDistance(cutoff * unit.nanometer)
    return hb, yu, ah

def set_harmonic_network(N,dmap,pae_inv,yu,ah,ssdomains=None,cs_cutoff=0.9,k_restraint=700.):
    cs = openmm.openmm.HarmonicBondForce()
    for i in range(N-2):
        for j in range(i+2,N):
            if ssdomains != None:  # use fixed domain boundaries for network
                ss = False
                for ssdom in ssdomains:
                    if i+1 in ssdom and j+1 in ssdom:
                        ss = True
                if ss:  # both residues in structured domains
                    if dmap[i,j] < cs_cutoff:  # nm
                        cs.addBond(i, j, dmap[i, j] * unit.nanometer,
                                   k_restraint * unit.kilojoules_per_mole / (unit.nanometer ** 2))
                        yu.addExclusion(i, j)
                        ah.addExclusion(i, j)
            elif isinstance(pae_inv, np.ndarray):  # use alphafold PAE matrix for network
                k = k_restraint * pae_inv[i,j]**2
                if k > 0.0:
                    cs.addBond(i,j, dmap[i,j]*unit.nanometer,
                                k*unit.kilojoules_per_mole/(unit.nanometer**2))
                    yu.addExclusion(i, j)
                    ah.addExclusion(i, j)
            else:
                raise
    cs.setUsesPeriodicBoundaryConditions(True)
    return cs, yu, ah

def simulate(config):
    """ Simulate CALVADOS using OpenMM

        * config is a dictionary """

    # parse config
    protein_name, temp, ionic = config['protein_name'], config['temp'], config['ionic']
    cutoff, steps, wfreq = config['cutoff'], config['steps'], config['wfreq']
    L = config['L']
    domain_resNum = config['domain_resNum']
    eps_factor = config["eps_factor"]
    pH = config["pH"]
    isIDP = config['isIDP']
    CoarseGrained = config['CoarseGrained']
    seq = config['seq']
    replica = 0
    use_pdb, path2pdb = config['use_pdb'], config['path2pdb']
    CALVADOS_version = config['CALVADOS_version']
    center_of_mass = np.array([0, 0, 0])
    if use_pdb:
        use_hnetwork = config['use_hnetwork']
        if use_hnetwork:
            k_restraint = config['k_restraint']
            use_ssdomains = config['use_ssdomains']
            if use_ssdomains:
                ssdomains = domain_resNum
                pae = None
            else:
                raise Exception("please make sure 'use_ssdomains' is True")
    else:
        path2pdb = ''
        use_hnetwork = False
        pae = None
        ssdomains = None
    # load residue parameters
    residues = load_parameters(protein_name, CALVADOS_version)
    # build protein dataframe
    df = pd.DataFrame(columns=['pH', 'ionic', 'temp', 'eps_factor', 'fasta'], dtype=object)
    df.loc[protein_name] = dict(pH=pH, ionic=ionic, temp=temp, eps_factor=eps_factor, fasta=seq)
    prot = df.loc[protein_name]
    print("pH:", prot.pH)
    print(f"Ionic strength: {prot.ionic} M")
    print(f"Temperature: {prot.temp} K")
    # LJ and YU parameters
    lj_eps, fasta, types, MWs = genParamsLJ(residues, protein_name, prot)
    # print("lj_eps:", lj_eps * unit.kilojoules_per_mole)
    yukawa_eps, yukawa_kappa = genParamsDH(residues, protein_name, prot, temp)

    N = len(fasta)  # number of residues
    n_chains = 1
    Lz = L

    # get input geometry

    if use_pdb:
        print(f'Starting from pdb structure {path2pdb}')
        pos, center_of_mass = geometry_from_pdb(protein_name, path2pdb, CoarseGrained=CoarseGrained, ssdomains=domain_resNum)
    else:
        spiral = True
        if spiral:
            pos = xy_spiral_array(N)
        else:
            pos = [[L / 2, L / 2, L / 2 + (i - N / 2.) * .38] for i in range(N)]
        pos = np.array(pos)

    top = build_topology(fasta, n_chains=n_chains)
    md.Trajectory(pos + center_of_mass, top, 0, [L, L, Lz], [90, 90, 90]).save_pdb(f'{protein_name}_tmp/ini_beads.pdb', force_overwrite=True)
    pos = pos + np.array([L / 2, L / 2, L / 2])
    a = md.Trajectory(pos, top, 0, [L, L, Lz], [90, 90, 90])
    a.save_pdb(f'{protein_name}_tmp/top_production.pdb', force_overwrite=True)
    # build openmm system
    system = openmm.System()

    # box
    a, b, c = build_box(L, L, Lz)
    system.setDefaultPeriodicBoxVectors(a, b, c)

    # load topology into system
    pdb = app.pdbfile.PDBFile(f'{protein_name}_tmp/top_production.pdb')

    # print(pdb.topology)

    # particles and termini
    system = add_particles(system, residues, prot, n_chains=n_chains)
    dmap = euclidean(pos, pos)
    # interactions
    hb, yu, ah = set_interactions(system, residues, prot, lj_eps, cutoff, yukawa_kappa, yukawa_eps, N,
                                  n_chains=n_chains, CoarseGrained=CoarseGrained, dismatrix=dmap, isIDP=isIDP,
                                  domain_resNum=domain_resNum)
    system.addForce(hb)
    system.addForce(yu)
    system.addForce(ah)

    # print("use_hnetwork: ", use_hnetwork)
    # harmonic network (currently unavailable for slab sim)
    if use_hnetwork:
        cs, yu, ah = set_harmonic_network(N, dmap, pae, yu, ah, ssdomains=ssdomains, k_restraint=k_restraint)
        # print(f"k_restraint used: {k_restraint}")
        system.addForce(cs)

    open(f'{protein_name}/system.xml', 'w').write(openmm.XmlSerializer.serialize(system))
    # use langevin integrator
    integrator = openmm.openmm.LangevinMiddleIntegrator(temp * unit.kelvin, 0.01 / unit.picosecond, 0.01 * unit.picosecond)
    # print(integrator.getFriction(), integrator.getTemperature())

    try:
        platform = openmm.Platform.getPlatformByName('CUDA')
        simulation = app.simulation.Simulation(pdb.topology, system, integrator, platform, dict(CudaPrecision='mixed'))
        print("Using GPU")
    except openmm.OpenMMException:
        platform = openmm.Platform.getPlatformByName('CPU')
        simulation = app.simulation.Simulation(pdb.topology, system, integrator, platform)
        print("Using CPU")

    simulation.context.setPositions(pdb.positions)
    simulation.minimizeEnergy()
    simulation.reporters.append(app.dcdreporter.DCDReporter(f'{protein_name}_tmp/production.dcd', wfreq, enforcePeriodicBox=False, append=False))

    simulation.reporters.append(
        app.statedatareporter.StateDataReporter(f'{protein_name}_tmp/statedata_production.log', int(wfreq),
                                                step=True, speed=True, elapsedTime=True, separator='\t', progress=True,
                                                remainingTime=True, totalSteps=steps))
    simulation.reporters.append(
        app.checkpointreporter.CheckpointReporter(file=f"{protein_name}_tmp/checkpoint_production.chk",
                                                  reportInterval=wfreq))


    starttime = time.time()  # begin timer
    # assert steps%wfreq==0
    print(f"Total frames: {steps//wfreq}; the first 10 frames will be discarded")
    for _ in progress_bar(list(range(steps//wfreq))):
        simulation.step(wfreq)
    endtime = time.time()  # end timer
    target_seconds = endtime - starttime  # total used time
    print(
        f"{protein_name} total simulation time: {target_seconds // 3600}h {(target_seconds // 60) % 60}min {np.round(target_seconds % 60, 2)}s")

def fix_topology(dcd,pdb):
    """
    Changes atom names to CA; even if CG-strategy is COM, it is still CA for convenience
    """
    t = md.load_dcd(dcd,pdb)
    cgtop = md.Topology()
    cgchain = cgtop.add_chain()
    for atom in t.top.atoms:
        cgres = cgtop.add_residue(atom.name, cgchain)
        cgtop.add_atom('CA', element=md.element.carbon, residue=cgres)
    traj = md.Trajectory(t.xyz, cgtop, t.time, t.unitcell_lengths, t.unitcell_angles)
    traj = traj.superpose(traj, frame=0)
    return traj

def centerDCD(config):
    protein_name = config["protein_name"]
    discard_first_nframes = config["discard_first_nframes"]
    CALVADOS_version = config["CALVADOS_version"]
    seq = config["seq"]
    residues = load_parameters(protein_name, CALVADOS_version)
    top = md.Topology()
    chain = top.add_chain()
    for resname in seq:
        residue = top.add_residue(residues.loc[resname, 'three'], chain)
        top.add_atom(residues.loc[resname, 'three'], element=md.element.carbon, residue=residue)
    for i in range(len(seq) - 1):
        top.add_bond(top.atom(i), top.atom(i + 1))
    traj = md.load_dcd(f"{protein_name}_tmp/production.dcd", top=top)[discard_first_nframes:]
    traj = traj.image_molecules(inplace=False, anchor_molecules=[set(traj.top.chain(0).atoms)], make_whole=True)
    traj.center_coordinates()
    traj.xyz += traj.unitcell_lengths[0, 0] / 2
    # print(f'Number of frames: {traj.n_frames}')
    traj.save_dcd(f'{protein_name}/{protein_name}_cg.dcd')
    traj[0].save_pdb(f'{protein_name}/{protein_name}_cg.pdb')
    traj = fix_topology(f'{protein_name}/{protein_name}_cg.dcd', f'{protein_name}/{protein_name}_cg.pdb')
    traj = traj.superpose(traj[0])
    traj[-1].save_pdb(f'{protein_name}/{protein_name}_fixed_cg.pdb')

In [ ]:
#@title <b><font color='#A79AB2'>Simulation analysis toolbox</font></b>
from scipy.optimize import curve_fit
from scipy import stats
from statsmodels.stats.weightstats import DescrStatsW


def calc_rg(t, protein_name, CALVADOS_version):
    df = load_parameters(protein_name, CALVADOS_version).set_index("three")
    masses = df.loc[[res.name for res in t.top.atoms],'MW'].values
    masses[0] += 2
    masses[-1] += 16
    # calculate the center of mass
    cm = np.sum(t.xyz*masses[np.newaxis,:,np.newaxis],axis=1)/masses.sum()
    # calculate residue-cm distances
    si = np.linalg.norm(t.xyz - cm[:,np.newaxis,:],axis=2)
    # calculate rg
    rgarray = np.sqrt(np.sum(si**2*masses,axis=1)/masses.sum())
    return rgarray

def calc_nu(traj,w=None):
    pairs = traj.top.select_pairs('all','all')
    d = md.compute_distances(traj,pairs)
    dmax = np.max(d)
    nres = traj.n_atoms
    ij = np.arange(2,nres,1)
    diff = [x[1]-x[0] for x in pairs]
    dij = np.empty(0)
    for i in ij:
        dij = np.append(dij,np.sqrt(np.average(np.mean(d[:,diff==i]**2,axis=1),weights=w,axis=0)))
    f = lambda x,R0,v : R0*np.power(x,v)
    popt, pcov = curve_fit(f,ij[ij>5],dij[ij>5],p0=[.4,.5])
    nu = popt[1]
    nu_err = pcov[1,1]**0.5
    R0 = popt[0]
    R0_err = pcov[0,0]**0.5
    return ij,dij,dmax,nu,nu_err,R0,R0_err

def calc_contact_map(t):
    indices = t.top.select_pairs('all','all')
    mask = np.abs(indices[:,0]-indices[:,1])>1 #exclude bonded pairs
    indices = indices[mask]
    d = md.compute_distances(t,indices) # distances between pairs for each frame
    cmap = np.nanmean((.5-.5*np.tanh((d-1)/.3)),axis=0)
    df_cmap = pd.DataFrame(index=range(t.n_atoms),columns=range(t.n_atoms),dtype=float)
    for k,(i,j) in enumerate(indices):
        df_cmap.loc[i,j] = cmap[k]
        df_cmap.loc[j,i] = cmap[k]
    return df_cmap

def calc_ree(t):
    return md.compute_distances( t, atom_pairs=np.array([[ 0,  len(list(t.top.atoms))-1]]) )[...,0]

def kde(a, w=None, phi_eff=None, min_=None, max_=None):
    if type(w) == 'NoneType':
        w = np.full(len(a), 1)
    if min_ == None:
        min_ = np.min(a)
    if max_ == None:
        max_ = np.max(a)
    x = np.linspace( min_, max_, num = 50 )
    d = stats.gaussian_kde( a, bw_method = "silverman", weights = w ).evaluate(x)
    u = DescrStatsW( a, weights = w )
    n_eff = phi_eff*a.size if phi_eff!=None else a.size
    return x,d/np.sum(d),u.mean,u.std/np.sqrt(n_eff)

def autoblock(cv, multi=1, plot=False):
    block = BlockAnalysis(cv, multi=multi)
    block.SEM()
    if plot == True:
        plt.errorbar(block.stat[...,0], block.stat[...,1], block.stat[...,2], fmt='', color='k', ecolor='0.5')
        plt.scatter(block.bs, block.sem,zorder=10,c='tab:red')
        plt.xlabel('Block size')
        plt.ylabel('SEM')
        plt.show()
    return block.av, block.sem, block.bs

def plot_dist(ax,x,p,av,color='k'):
    ax.plot(x,p,c=color)
    ax.axvline(av,color=color)
    ax.set_xlim(np.min(x), np.max(x))
    ax.set_ylim(0,np.max(p)*1.2)

def plot_rew_dist(ax,x,p,av):
    ax.plot(x,p,c='tab:red')
    ax.axvline(av,0,100, color='tab:red')
    ax.set_xlim(np.min(x), np.max(x))
    ax.set_ylim(0,np.max(p)*1.2)

def error_ratio(v1,v2,e1,e2):
    ratio = v1/v2
    return ratio*np.sqrt((e1/v1)**2+(e2/v2)**2)

In [ ]:
# @title 5. Run MD simulation
config = yaml.safe_load(open(f'{name}/config_sim.yaml', 'r'))
simulate(config)
centerDCD(config)

In [ ]:
# @title Install Pepsi-SAXS and download BLOCKING, BME, and BIFT
%%bash

wget https://files.inria.fr/NanoDFiles/Website/Software/Pepsi-SAXS/Linux/3.0/Pepsi-SAXS-Linux.zip &> /dev/null
unzip Pepsi-SAXS-Linux.zip &> /dev/null
rm Pepsi-SAXS-Linux.zip

wget https://raw.githubusercontent.com/fpesceKU/BLOCKING/main/block_tools.py &> /dev/null
wget https://raw.githubusercontent.com/fpesceKU/BLOCKING/main/main.py &> /dev/null

wget https://raw.githubusercontent.com/KULL-Centre/BME/main/BME_tools.py &> /dev/null
wget https://raw.githubusercontent.com/KULL-Centre/BME/main/BME.py &> /dev/null

wget https://raw.githubusercontent.com/ehb54/GenApp-BayesApp/main/bin/source/bift.f &> /dev/null
gfortran bift.f -march=native -O2 -o bift

In [ ]:
# @title 6. Analysis
from main import BlockAnalysis
import matplotlib as mpl
import matplotlib.pyplot as plt

try:
    os.mkdir(f'{config["protein_name"]}/analyses')
except:
    pass

config = yaml.safe_load(open(f'{name:s}/config_sim.yaml', 'r'))
t = md.load_dcd(f"{config['protein_name']}/{config['protein_name']}_cg.dcd",
                f"{config['protein_name']}/{config['protein_name']}_cg.pdb")

rg_array = calc_rg(t, config["protein_name"], config["CALVADOS_version"])
ree_array = calc_ree(t)
ij,dij,dmax,nu,nu_err,R0,R0_err = calc_nu(t)
df_cmap = calc_contact_map(t)

rg, rg_err, rg_blocksize = autoblock(rg_array)
x_rg, p_rg, _, _ = kde(rg_array)

ree, ree_err, ree_blocksize = autoblock(ree_array)
x_ree, p_ree, _, _ = kde(ree_array)

# Plot results
mpl.rcParams.update({'font.size': 6})
fig, axs = plt.subplots(2, 2, figsize=(4,3), facecolor='w', dpi=200)
axs = axs.flatten()

axs[0].plot(x_rg, p_rg)
top = p_rg.max()+0.1*p_rg.max()
axs[0].axvline(rg)

axs[0].set_xlabel(r'$R_g$ (nm)')
axs[0].set_ylabel(r'$p(R_g)$')
axs[0].set_ylim(0,top)
axs[0].fill_between([rg-rg_err,rg+rg_err],0,top,alpha=0.3)

axs[1].plot(x_ree, p_ree)
top = p_ree.max()+0.1*p_ree.max()
axs[1].vlines(ree,0,top)
axs[1].set_xlabel(r'$R_{ee}$ (nm)')
axs[1].set_ylabel(r'$p(R_{ee})$')
axs[1].set_ylim(0,top)

im = axs[2].imshow(df_cmap,extent=[1, df_cmap.shape[0], 2, df_cmap.shape[0]],origin='lower',
                   aspect='equal',vmin=0,vmax=.5,cmap=plt.cm.Blues)
cb = plt.colorbar(im, ax=axs[2], fraction=0.05, pad=0.04)
cb.set_label('Contacts', labelpad=-3)
cb.set_ticks([0,.5])
axs[2].set_xlabel('Residue #')
axs[2].set_ylabel('Residue #')

if config["isIDP"]:
    axs[3].plot(ij,dij)
    dij_fit = R0*np.power(ij,nu)
    axs[3].plot(ij, dij_fit,c='0.3',ls='dashed',label='Fit')
    axs[3].set_xlabel('$|i-j|$')
    axs[3].set_ylabel(r'$\sqrt{\langle R_{ij}^2 \rangle}$ (nm)')
    axs[3].text(0.05, 0.9, r'$\nu$={:.2f}'.format(nu), horizontalalignment='left',
                verticalalignment='center', transform=axs[3].transAxes, fontsize=6)
    axs[3].legend(loc='lower right')
else:
    axs[3].axis('off')

plt.tight_layout(w_pad=3)

plt.savefig(f'{config["protein_name"]}/analyses/conformational_properties.pdf',
            dpi=300, facecolor='w', edgecolor='w', orientation='portrait',
            bbox_inches='tight')
plt.show()

if config["isIDP"]:
    df_means = pd.DataFrame(data=np.c_[[rg,rg_err],[ree,ree_err],[nu,nu_err]],
                        columns=['<Rg> (nm)','<Ree> (nm)','nu'],
                        index=['Value','Error'])
else:
    df_means = pd.DataFrame(data=np.c_[[rg,rg_err],[ree,ree_err]],
                        columns=['<Rg> (nm)','<Ree> (nm)'],
                        index=['Value','Error'])

df_means.to_csv(f'{config["protein_name"]}/analyses/conf_properties.csv')
df_cmap.to_csv(f'{config["protein_name"]}/analyses/contact_map.csv')

df_means

In [ ]:
# @title 7. Visualize trajectory
import py3Dmol
class Atom(dict):
    def __init__(self, line):
        self["type"] = line[0:6].strip()
        self["idx"] = line[6:11].strip()
        self["name"] = line[12:16].strip()
        self["resname"] = line[17:20].strip()
        self["resid"] = int(int(line[22:26]))
        self["x"] = float(line[30:38])
        self["y"] = float(line[38:46])
        self["z"] = float(line[46:54])
        self["sym"] = line[76:78].strip()
    def __str__(self):
        line = list(" " * 80)

        line[0:6] = self["type"].ljust(6)
        line[6:11] = self["idx"].ljust(5)
        line[12:16] = self["name"].ljust(4)
        line[17:20] = self["resname"].ljust(3)
        line[22:26] = str(self["resid"]).ljust(4)
        line[30:38] = str(self["x"]).rjust(8)
        line[38:46] = str(self["y"]).rjust(8)
        line[46:54] = str(self["z"]).rjust(8)
        line[76:78] = self["sym"].rjust(2)
        return "".join(line) + "\n"

class Molecule(list):
    def __init__(self, file):
        for line in file:
            if "ATOM" in line or "HETATM" in line:
                self.append(Atom(line))

    def __str__(self):
        outstr = ""
        for at in self:
            outstr += str(at)

        return outstr

if not config["isIDP"]:
    all_domain_residues = []
    for temp in domain_resNum:
        all_domain_residues += temp
    template = Molecule(open(f'{config["protein_name"]}/{config["protein_name"]}_cg.pdb', 'r'))
    for at in template:
        if int(at["idx"]) in all_domain_residues:
            at["pymol"] = {"sphere": {'color': "#009988"}}

md.load_dcd(f'{config["protein_name"]}/{config["protein_name"]}_cg.dcd',
            f'{config["protein_name"]}/{config["protein_name"]}_cg.pdb').save_pdb(
                f'{config["protein_name"]}/{config["protein_name"]}_view.pdb')
traj = md.load_pdb(f'{config["protein_name"]}/{config["protein_name"]}_view.pdb')

if not config["isIDP"]:
    domain_resNum = config["domain_resNum"]
    traj = traj.superpose(traj[0], atom_indices=np.array(domain_resNum[0]))
    traj.save_pdb(f'{config["protein_name"]}/{config["protein_name"]}_view.pdb')
with open(f'{config["protein_name"]}/{config["protein_name"]}_view.pdb') as ifile:
    view = py3Dmol.view(width=400, height=300)
    view.addModelsAsFrames("".join([x for x in ifile]))
    if not config["isIDP"]:
        for i, at in enumerate(template):
            default = {"sphere": {'color': 'EE7733'}}
            view.setStyle({'model': -1, 'serial': i+1}, at.get("pymol", default))
    else:
        view.setStyle({'model': -1}, {"sphere": {'color': '#EE7733'}})
    view.zoomTo()
    view.animate({'loop': "forward", 'reps': 1, 'interval':1000})
    view.show()

In [ ]:
# @title Install cg2all for all-atom reconstruction (will take ~5 mins)
%%bash
pip install dgl -f https://data.dgl.ai/wheels/torch-2.3/cu121/repo.html &> /dev/null
pip install -q git+http://github.com/huhlim/cg2all@cuda-12 &> /dev/null
pip install -q py3Dmol gdown mrcfile &> /dev/null

In [ ]:
#@title Reconstruct all-atom trajectory
import locale
locale.getpreferredencoding = lambda: "UTF-8"
import yaml
config = yaml.safe_load(open(f'{name:s}/config_sim.yaml', 'r'))
protein_name = config["protein_name"]
print(f'Backmapping {protein_name} to all-atom',end=' ')
isIDP = config["isIDP"]
cg_model = "CalphaBasedModel" if isIDP else "ResidueBasedModel"
print(f"using {cg_model}")
!convert_cg2all -p {protein_name}/{protein_name}_fixed_cg.pdb -d {protein_name}/{protein_name}_cg.dcd -o {protein_name}/traj_AA.dcd -opdb {protein_name}/top_AA.pdb --cg {cg_model}
# from openmm.app import PDBFile, ForceField
# from openmm.unit import nanometer, picoseconds, kelvin
# from openmm import LangevinMiddleIntegrator
from openmm.app import *
from openmm import *
from openmm.unit import *

top_AA_em = md.load_pdb(f'{config["protein_name"]}/top_AA.pdb')
translated_em = top_AA_em.xyz[0]-np.mean(top_AA_em.xyz[0],axis=0)
L_em = np.ceil(np.max([translated_em[:,0].max()-translated_em[:,0].min(),
                       translated_em[:,1].max()-translated_em[:,1].min(),
                       translated_em[:,2].max()-translated_em[:,2].min()]))+2
md.Trajectory(translated_em, top_AA_em.top, 0, [L_em, L_em, L_em], [90, 90, 90]).save_pdb(f'{config["protein_name"]}/top_AA.pdb')
pdb_em = PDBFile(f'{config["protein_name"]}/top_AA.pdb')
top_AA_em = md.load_pdb(f'{config["protein_name"]}/top_AA.pdb')
forcefield_em = ForceField('amber14-all.xml', 'amber14/tip3pfb.xml')
system_em = forcefield_em.createSystem(pdb_em.topology, nonbondedMethod=PME, nonbondedCutoff=1*nanometer, constraints=HBonds)
integrator_em = LangevinMiddleIntegrator(300*kelvin, 1/picosecond, 0.004*picoseconds)
try:
    simulation_em = Simulation(pdb_em.topology, system_em, integrator_em, openmm.Platform.getPlatformByName("CUDA"))
except OpenMMException:
    simulation_em = Simulation(pdb_em.topology, system_em, integrator_em, openmm.Platform.getPlatformByName("CPU"))
simulation_em.context.setPositions(pdb_em.positions)
simulation_em.minimizeEnergy()
state_em = simulation_em.context.getState(getPositions=True)
pos_em = state_em.getPositions(asNumpy=True)
md.Trajectory(pos_em._value, top_AA_em.top, 0, top_AA_em.unitcell_lengths, top_AA_em.unitcell_angles).save_pdb(f'{config["protein_name"]}/top_AA.pdb')

In [ ]:
# @title 8. Download results

# @markdown In this zip file:

# @markdown `${protein_name}_cg.dcd`: the coarse-grained trajectory file;

# @markdown `${protein_name}_cg.pdb`: the coarse-grained topology file;

# @markdown After reconstruction

# @markdown `$traj_AA.dcd`: the reconstructed trajectory file;

# @markdown `$top_AA.pdb`: the all-atom topology file;

# @markdown Plots and analyses

# @markdown `${protein_name}/analyses/conformational_properties.pdf`;

# @markdown `${protein_name}/analyses/conf_properties.csv`;

# @markdown `${protein_name}/analyses/contact_map.csv`;

files_to_download = [f'{config["protein_name"]}/{config["protein_name"]}_cg.dcd',
                     f'{config["protein_name"]}/{config["protein_name"]}_fixed_cg.pdb',
                     f'{config["protein_name"]}/{config["protein_name"]}_cg.pdb',
                     f'{config["protein_name"]}/traj_AA.dcd',
                     f'{config["protein_name"]}/top_AA.pdb',
                     f'{config["protein_name"]}/analyses',
                     f'{config["protein_name"]}/config_sim.yaml',
                     'residues.csv']

import glob
import subprocess

for filename in glob.glob(f'{config["protein_name"]}/*'):
    if filename not in files_to_download:
        try:
            os.remove(f'{filename:s}')
        except:
            shutil.rmtree(f'{filename:s}')

zipper = f'zip -r {config["protein_name"]}.zip {config["protein_name"]}'
subprocess.run(zipper.split())
files.download(f'{config["protein_name"]}.zip')